# Computational Performance -- one call per method, one synthetic example

Same benchmark methodology as `Rupture_Validation.ipynb`, run on one synthetic
series (`noise_red_snr3.npz`, seed 0) instead of NGRIP. One fixed parameter set
per method, using the actual tuned values this dataset already converges to
(`pen=2.169` / `n_bkps=45`, from the existing `*_grid_red_snr3.ipynb` results)
rather than an arbitrary placeholder -- so the timing below reflects a realistic
call, not a toy one. A different cost function changes this; noted in the plan
as a caveat, not re-derived here.

In [11]:
import gc
import os
import time
import threading

import numpy as np
import pandas as pd
import psutil

import ammonyte as amt

In [12]:
def benchmark_function(func, sample_interval=0.01):
    """
    Measure wall time, CPU time, and process peak RSS memory.

    RSS is sampled from the OS via psutil, so it captures memory used by
    native libraries (e.g. SciPy/LAPACK), not just Python-level allocations.
    """
    process = psutil.Process(os.getpid())
    gc.collect()

    baseline_rss = process.memory_info().rss
    peak_rss = baseline_rss
    stop_sampling = threading.Event()

    def sample_memory():
        nonlocal peak_rss
        while not stop_sampling.is_set():
            rss = process.memory_info().rss
            peak_rss = max(peak_rss, rss)
            stop_sampling.wait(sample_interval)

    sampler = threading.Thread(target=sample_memory, daemon=True)

    start_wall = time.perf_counter()
    start_cpu = time.process_time()
    sampler.start()

    try:
        output = func()
    finally:
        cpu_time = time.process_time() - start_cpu
        wall_time = time.perf_counter() - start_wall
        stop_sampling.set()
        sampler.join()

    metrics = {
        'Wall Time (s)': wall_time,
        'CPU Time (s)': cpu_time,
        'Average CPU (%)': 100 * cpu_time / wall_time,
        'Peak Memory (MB)': (peak_rss - baseline_rss) / 1024**2,
    }
    return output, metrics

## Load one synthetic example (red/snr3, seed 0)

In [13]:
d = np.load('../noise_sensitivity_data/noise_red_snr3.npz')
series = amt.Series(time=d['time'], value=d['db_noisy'][0], label='bench', verbose=False)
print(f"{len(d['time'])} points")

6001 points


## Ruptures methods (Pelt, Binseg, BottomUp, Window, KernelCPD, Dynp)

In [18]:
algorithms_perf = ['Pelt', 'Binseg', 'BottomUp', 'Window', 'KernelCPD', 'Dynp']
performance = []

# pen=2.169 / n_bkps=45 -- the actual tuned values this dataset converges to
# (Methods/*/results/*_grid_red_snr3_conclusion.csv), not an arbitrary pen=10.
# 45 also matches this seed's real number of ground-truth transitions (47).
for algo in algorithms_perf:
    print(f"\nBenchmarking {algo}")

    def run_algo(algo=algo):
        if algo == 'Dynp':
            # jump=5: n_bkps=45 at the ruptures default jump=1 does not finish in
            # 15+ minutes for a single call (confirmed earlier) -- jump=5 keeps
            # this call tractable while still using a realistic n_bkps.
            return series.ruptures(algo=algo, cost='l2', n_bkps=45, jump=1)
        elif algo == 'Window':
            return series.ruptures(algo=algo, cost='l2', pen=0.0001, width=80)
        elif algo == 'KernelCPD':
            return series.ruptures(algo=algo, cost='linear', pen=2.169)
        else:
            return series.ruptures(algo=algo, cost='l2', pen=2.169)

    transitions_perf, metrics = benchmark_function(run_algo)
    detected_perf = transitions_perf.jump_times[~np.isnan(transitions_perf.jump_times)]

    performance.append({
        'Algorithm': algo,
        **metrics,
        'Detections': len(detected_perf),
    })

    print(f"Completed in {metrics['Wall Time (s)']:.3f}s "
          f"(CPU: {metrics['CPU Time (s)']:.3f}s, Peak Memory: {metrics['Peak Memory (MB)']:.1f} MB)")


Benchmarking Pelt
Completed in 7.231s (CPU: 7.212s, Peak Memory: 5.0 MB)

Benchmarking Binseg
Completed in 0.770s (CPU: 0.770s, Peak Memory: 0.8 MB)

Benchmarking BottomUp
Completed in 0.079s (CPU: 0.079s, Peak Memory: 0.7 MB)

Benchmarking Window
Completed in 0.175s (CPU: 0.169s, Peak Memory: 0.3 MB)

Benchmarking KernelCPD
Completed in 0.007s (CPU: 0.007s, Peak Memory: 0.0 MB)

Benchmarking Dynp
Completed in 3342.239s (CPU: 3087.456s, Peak Memory: 5084.1 MB)


## KS test

In [19]:
print("\nBenchmarking KS_test")

def run_ks():
    return series.kstest(w_min=0.2, w_max=1, n_w=10, n_c=5, s_c=1.5, d_c=0.65, x_c=None)

transitions_perf, metrics = benchmark_function(run_ks)
detected_perf = transitions_perf.jump_times[~np.isnan(transitions_perf.jump_times)]

performance.append({
    'Algorithm': 'KS_test',
    **metrics,
    'Detections': len(detected_perf),
})

print(f"Completed in {metrics['Wall Time (s)']:.3f}s "
      f"(CPU: {metrics['CPU Time (s)']:.3f}s, Peak Memory: {metrics['Peak Memory (MB)']:.1f} MB)")


Benchmarking KS_test
Completed in 8.170s (CPU: 8.123s, Peak Memory: 0.6 MB)


## LERM

In [20]:
print("\nBenchmarking LERM")

def run_lerm():
    embedded = amt.TimeEmbeddedSeries(series, m=5, tau=None)
    eps_result = embedded.find_epsilon(eps=1, target_density=0.05, tolerance=0.01, amp=30)
    rm = eps_result['Output']
    fi_series = rm.laplacian_eigenmaps(w_size=20, w_incre=4)
    smoothed = amt.utils.fisher.smooth_series(fi_series, block_size=2)
    return smoothed.lerm_transitions()

transitions_perf, metrics = benchmark_function(run_lerm)
detected_perf = transitions_perf.jump_times[~np.isnan(transitions_perf.jump_times)]

performance.append({
    'Algorithm': 'LERM',
    **metrics,
    'Detections': len(detected_perf),
})

print(f"Completed in {metrics['Wall Time (s)']:.3f}s "
      f"(CPU: {metrics['CPU Time (s)']:.3f}s, Peak Memory: {metrics['Peak Memory (MB)']:.1f} MB)")


Benchmarking LERM
Initial density is 0.5473
Initial density is not within the tolerance window, searching...
Epsilon: 0.0000, Density: 0.0000
Epsilon: 0.0000, Density: 0.0000
Epsilon: 0.0000, Density: 0.0000
Epsilon: 0.3750, Density: 0.0407
Epsilon: 0.3750, Density: 0.0407.
Completed in 22.735s (CPU: 25.911s, Peak Memory: 0.8 MB)


## Summary table

In [21]:
df_perf = pd.DataFrame(performance)
print("\n" + "="*80)
print("COMPUTATIONAL PERFORMANCE COMPARISON")
print("="*80)
print(df_perf.round(3).to_markdown(index=False))
print("="*80)
df_perf


COMPUTATIONAL PERFORMANCE COMPARISON
| Algorithm   |   Wall Time (s) |   CPU Time (s) |   Average CPU (%) |   Peak Memory (MB) |   Detections |
|:------------|----------------:|---------------:|------------------:|-------------------:|-------------:|
| Pelt        |           7.231 |          7.212 |            99.74  |              4.984 |           43 |
| Binseg      |           0.77  |          0.77  |           100.1   |              0.766 |           42 |
| BottomUp    |           0.079 |          0.079 |           100.029 |              0.656 |           44 |
| Window      |           0.175 |          0.169 |            96.166 |              0.312 |           56 |
| KernelCPD   |           0.007 |          0.007 |            99.892 |              0.031 |           43 |
| Dynp        |        3342.24  |       3087.46  |            92.377 |           5084.06  |           45 |
| KS_test     |           8.17  |          8.123 |            99.425 |              0.562 |          203 |

,Algorithm,Wall Time (s),CPU Time (s),Average CPU (%),Peak Memory (MB),Detections
0,Pelt,7.231039,7.212216,99.739689,4.984375,43
1,Binseg,0.769686,0.770454,100.099765,0.765625,42
2,BottomUp,0.079007,0.079030,100.029323,0.656250,44
3,Window,0.175342,0.168619,96.165528,0.312500,56
4,KernelCPD,0.006781,0.006774,99.891849,0.031250,43
5,Dynp,3342.239456,3087.455906,92.376861,5084.062500,45
6,KS_test,8.170167,8.123168,99.424750,0.562500,203
7,LERM,22.734951,25.910512,113.967749,0.750000,26


# Plan: Full Grid Search / Optimization Across All 9 Noise Conditions

Built from the benchmark table above (real, measured numbers, `red/snr3` seed 0)
plus the earlier per-method `*_grid_red_snr3.ipynb` notebooks already validated
on that one condition. Two parts: a **main workflow** (fixed, happens the same
way regardless of the choices below) and a set of **options** (real decisions,
not locked in yet).

**Scope**: 9 noise combinations -- `{white, pink, red} x {snr2, snr3, snr4}`
. 8 methods: Pelt, Binseg,
BottomUp, Window, KernelCPD, Dynp, KS test, LERM. `N_JOBS = 24` (actual CARC
core allocation).

## Main workflow

**1. Submission -- one SLURM array per method, one array index per noise condition.**

A SLURM array lets one `sbatch` submission launch many near-identical jobs at
once, each getting a different `$SLURM_ARRAY_TASK_ID`. Instead of hand-submitting
9 separate jobs per method, submit one `.slurm` file with `#SBATCH --array=0-8`
and CARC runs 9 independent copies, IDs 0-8. A small lookup table maps each ID
to a `(color, level)` pair:

```python
COMBOS = [
    ('white','snr2'), ('white','snr3'), ('white','snr4'),
    ('pink','snr2'),  ('pink','snr3'),  ('pink','snr4'),
    ('red','snr2'),   ('red','snr3'),   ('red','snr4'),
]
```

```bash
#SBATCH --array=0-8
...
python pelt_grid.py --task-id $SLURM_ARRAY_TASK_ID
```
```python
color, level = COMBOS[task_id]
```

One `sbatch pelt_grid.slurm` -> 9 independent jobs, each on a different noise
file, each with its own log and walltime. A stuck/failed condition is
resubmitted alone (`sbatch --array=3 pelt_grid.slurm`) without touching the
other 8. Done once per method: **8 scripts, 8 `sbatch` submissions, 72 job
runs total** -- not 72 files.

**2. Load** the one `.npz` for that job's `(color, level)`.

**3. Seed pool + train/test split**:
- Standard budget: pool 20 -> 14 train / 6 test seeds (most cost functions).
- Reduced budget: pool ~10 -> 7 train / 3 test seeds (expensive cost functions,
  and Dynp overall).

**4. Parameter search** -- evaluate every parameter point (grid or LHS, see
Options) against every train seed, scored by F1. `min_size` fixed to
`[5, 30, 50]` everywhere (empirically the search always picks the smallest
value regardless of range, so it's checked at 3 points).

**5. Select** the single parameter point with highest mean train F1.

**6. Evaluate once** on the held-out test seeds with that winning combo.

**7. Write output**: `raw.csv` (every point x seed), `summary.csv` (aggregated
per point), `conclusion.csv` (winner + test metrics), plus a plot (F1 surface
for grid search, parallel-coordinates for LHS).

**8. Aggregate** all 72 completed runs into one master table (method x color x
level x cost), extending `compare_methods.ipynb`'s existing `load_all()`.

## Options -- real choices, not locked in yet

**A. Dynp resolution: `jump=5` vs `jump=1`.**

| | `jump=5` | `jump=1` |
|---|---|---|
| Per-call time | 86.6 s | 3,342 s (55.7 min) |
| Per-call peak memory | 216 MB | 5.08 GB |
| At 24 parallel workers | ~5.2 GB | **~122 GB -- most of a node** |
| Full 72-combo grid | ~30 min/condition | not realistic as a grid -- only usable as a single confirmatory run on the already-chosen winning combo |

`jump=5` is lighter and is what makes the full grid affordable; `jump=1` stays
available as an optional, single-combo precision check after the grid picks a
winner, not as the grid's default resolution.

**B. Search method per method group -- full grid vs. Latin Hypercube Sampling
(LHS, `scipy.stats.qmc.LatinHypercube`, no new dependency).**

LHS's real advantage here isn't just "fewer points than a full grid" -- it can
sample **continuously** within a parameter's range (e.g. `d_c ~ Uniform(0.5, 0.9)`
can land at `d_c=0.683`), rather than being restricted to a handful of
pre-chosen discrete values the way every grid parameter is today.


| Tier | Peak mem/call | x 24 workers |
|---|---|---|
| Pelt/Binseg/BottomUp/Window/KernelCPD/KS | <15 MB | <400 MB |
| LERM (cold cache -- the number to size against, not a warm-cache rerun) | ~950 MB | ~22.8 GB |
| Dynp, `jump=5` | 216 MB | ~5.2 GB |
| Dynp, `jump=1` (single-combo check only) | 5.08 GB | ~122 GB |